# Libraries


In [ ]:
import os
import warnings
from pathlib import Path

import h5py
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.patches import Patch, Rectangle
from scipy.ndimage import binary_dilation, gaussian_filter1d
from skimage.segmentation import find_boundaries

sns.set_theme(style="white", context="talk")
plt.rcParams["axes.grid"] = False


# Task Structure


In [ ]:
ALLEN_DOC_URL = "https://allenneuraldynamics.github.io/openscope-community-predictive-processing/stimuli/sensory-motor-closed-loop/"

oddball_map = {
    "omission": "omission",
    "motor_omission": "omission",
    "halt": "halt",
    "motor_halt": "halt",
    "motor_orientation_90": "orientation_90",
    "motor_orientation_45": "orientation_45",
}
oddball_order = ["omission", "halt", "orientation_90", "orientation_45"]
oddball_colors = {
    "omission": "#2e8b57",
    "halt": "#6a3d9a",
    "orientation_90": "#ff8c00",
    "orientation_45": "#d62728",
}

display_names = {
    "standard_control": "Control\n(Standard stimuli)",
    "motor_oddball": "Sensory-Motor\nMismatch",
    "sequential_control_block": "Control\n(Shuffled sequences)",
    "jitter_control": "Control\n(Duration tuning)",
    "open_loop_prerecorded": "Control\n(Pre-recorded)",
    "movie": "Movie",
    "rf_mapping": "RF Mapping",
}

block_colors = {
    "standard_control": "#d7e8f7",
    "motor_oddball": "#ffe0b2",
    "sequential_control_block": "#eeeeee",
    "jitter_control": "#f9e4d8",
    "open_loop_prerecorded": "#d7d8f7",
    "movie": "#d9d9d9",
    "rf_mapping": "#fff9b0",
}

base_oris = [0.0, 22.5, 45.0, 67.5, 90.0, 112.5, 135.0, 157.5]
palette = sns.color_palette("tab10", n_colors=len(base_oris))
ori_color_map = {ori: palette[i] for i, ori in enumerate(base_oris)}


def resolve_ops_path(session_dir):
    session_dir = Path(session_dir)
    candidates = [
        session_dir / "ops.npy",
        session_dir / "suite2p" / "plane0" / "ops.npy",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def session_file_flags(session_dir):
    session_dir = Path(session_dir)
    return {
        "ops": resolve_ops_path(session_dir).exists(),
        "dff": (session_dir / "dff.h5").exists() or (session_dir / "memmap" / "dff" / "dff.mmap").exists(),
        "raw_voltages": (session_dir / "raw_voltages.h5").exists() or (session_dir / "memmap" / "raw_voltages" / "vol_time.mmap").exists(),
        "stimulus_table": (session_dir / "stimulus_table.csv").exists(),
        "masks": (session_dir / "masks.h5").exists() or (session_dir / "masks.npy").exists() or (session_dir / "qc_results" / "masks.npy").exists(),
        "bpod_mat": (session_dir / "bpod_session_data.mat").exists(),
    }


def summarize_session_candidate(session_dir):
    flags = session_file_flags(session_dir)
    present = [name for name, ok in flags.items() if ok]
    missing = [name for name, ok in flags.items() if not ok and name in {"ops", "dff", "stimulus_table"}]
    return {
        "session_dir": Path(session_dir),
        "flags": flags,
        "present": present,
        "missing": missing,
    }


def find_mc19_session_dirs(search_roots=None, session_hint="MC19"):
    if search_roots is None:
        search_roots = [Path.home() / "MC19_Processed_New"]

    hint = (session_hint or "MC19").lower()
    roots = [Path(root).expanduser() for root in search_roots if Path(root).expanduser().exists()]
    candidates = []
    seen = set()
    for root in roots:
        for stim_path in root.rglob("stimulus_table.csv"):
            session_dir = stim_path.parent
            if hint not in str(session_dir).lower():
                continue
            resolved = session_dir.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            candidates.append(summarize_session_candidate(session_dir))
    candidates.sort(key=lambda c: str(c["session_dir"]))
    return candidates


def find_candidate_csvs(session_dir):
    session_dir = Path(session_dir)
    stim_candidates = sorted(session_dir.rglob("stimulus_table.csv"))
    voltage_candidates = sorted(session_dir.rglob("*VoltageRecording*.csv")) + sorted(session_dir.rglob("*voltage*.csv"))
    return stim_candidates, voltage_candidates


def discover_session_files(
    session_hint="MC19",
    search_roots=None,
    manual_session_dir=None,
    manual_stim_csv=None,
    manual_voltage_csv=None,
    session_index=-1,
):
    if search_roots is None:
        search_roots = [Path.home() / "MC19_Processed_New"]

    if manual_session_dir is not None:
        session_dir = Path(manual_session_dir).expanduser()
        candidate = summarize_session_candidate(session_dir)
    else:
        candidates = find_mc19_session_dirs(search_roots=search_roots, session_hint=session_hint)
        if not candidates:
            raise FileNotFoundError(
                "No processed MC19 sessions were found under the configured search roots. "
                f"Search roots: {search_roots}"
            )
        full_candidates = [c for c in candidates if not c["missing"]]
        if not full_candidates:
            summary = "\n".join(
                f"- {c['session_dir']} | missing: {', '.join(c['missing']) if c['missing'] else 'none'} | present: {', '.join(c['present'])}"
                for c in candidates[:10]
            )
            raise FileNotFoundError(
                "Found MC19 sessions, but none contain the required core files (ops + dff + stimulus_table).\nCandidates:\n" + summary
            )
        candidate = full_candidates[session_index]
        session_dir = candidate["session_dir"]

    stim_candidates, voltage_candidates = find_candidate_csvs(session_dir)
    stim_csv = Path(manual_stim_csv).expanduser() if manual_stim_csv else (stim_candidates[0] if stim_candidates else None)
    voltage_csv = Path(manual_voltage_csv).expanduser() if manual_voltage_csv else (voltage_candidates[0] if voltage_candidates else None)

    return {
        "session_dir": Path(session_dir),
        "ops_path": resolve_ops_path(session_dir),
        "dff_path": Path(session_dir) / "dff.h5",
        "raw_voltages_path": Path(session_dir) / "raw_voltages.h5",
        "bpod_mat_path": Path(session_dir) / "bpod_session_data.mat",
        "stim_csv": stim_csv,
        "voltage_csv": voltage_csv,
        "session_candidate": candidate,
        "stim_candidates": stim_candidates,
        "voltage_candidates": voltage_candidates,
    }


def ensure_numeric(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def add_time_columns(df):
    df = df.copy()
    df["Duration"] = pd.to_numeric(df["Duration"], errors="coerce")
    df["Time_sec"] = df["Duration"].shift(fill_value=0).cumsum()
    df["Stimulus_End_sec"] = df["Time_sec"] + df["Duration"].fillna(0)
    if "Orientation" in df.columns:
        df["Orientation_mod180"] = np.mod(df["Orientation"], 180)
    return df


def build_block_summary(df):
    summary = (
        df.groupby(["Block_Number", "Block_Label", "Block_Type"], dropna=False, as_index=False)
        .agg(
            start_sec=("Time_sec", "min"),
            end_sec=("Stimulus_End_sec", "max"),
            reported_duration_min=("Block_Duration_Minutes", "first"),
            n_trials=("Trial_Number", "count"),
        )
        .sort_values("Block_Number")
        .reset_index(drop=True)
    )
    summary["actual_duration_min"] = (summary["end_sec"] - summary["start_sec"]) / 60.0
    summary["duration_diff_min"] = summary["reported_duration_min"] - summary["actual_duration_min"]
    return summary


def format_axis(ax, title=None, xlabel=None, ylabel=None):
    if title is not None:
        ax.set_title(title, weight="bold", pad=12)
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    ax.grid(False)


def plot_task_structure(df, block_df, save_dir=None):
    plot_df = df[df["Block_Type"] != "movie"].copy()
    oddball_df = plot_df[plot_df["Trial_Type"].isin(oddball_map)].copy()
    oddball_df["Oddball_Type"] = oddball_df["Trial_Type"].map(oddball_map)

    ori_df = plot_df[
        plot_df["Trial_Type"].isin(
            ["single", "standard", "prerecorded", "motor_orientation_45", "motor_orientation_90", "rf_mapping"]
        )
    ].copy()
    ori_df = ori_df.dropna(subset=["Orientation"])

    fig = plt.figure(figsize=(24, 16))
    gs = gridspec.GridSpec(3, 1, height_ratios=[1.3, 1, 1], hspace=0.15)
    ax_block = fig.add_subplot(gs[0])
    ax_ori = fig.add_subplot(gs[1], sharex=ax_block)
    ax_odd = fig.add_subplot(gs[2], sharex=ax_block)

    for _, row in block_df.iterrows():
        x0 = row["start_sec"]
        width = row["end_sec"] - row["start_sec"]
        block_type = row["Block_Type"]
        rect = Rectangle((x0, 0), width, 1, facecolor=block_colors.get(block_type, "#dddddd"), edgecolor="black", linewidth=0.8, alpha=0.8)
        ax_block.add_patch(rect)
        label = (
            f"Block {int(row['Block_Number'])}\n"
            f"{display_names.get(block_type, row['Block_Label'])}\n"
            f"actual {row['actual_duration_min']:.2f} min\n"
            f"reported {row['reported_duration_min']:.2f} min"
        )
        ax_block.text(x0 + width / 2, 0.5, label, ha="center", va="center", fontsize=10, weight="bold", bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.8))

    ax_block.set_ylim(0, 1)
    ax_block.set_yticks([])
    format_axis(ax_block, title="Block Structure and Timeline", ylabel="Blocks")
    secax = ax_block.secondary_xaxis("top", functions=(lambda x: x / 60, lambda x: x * 60))
    secax.set_xlabel("Time (minutes)")

    for ori in sorted(ori_df["Orientation"].dropna().unique()):
        sub = ori_df[ori_df["Orientation"] == ori]
        color_key = float(np.mod(ori, 180))
        label = f"{int(color_key) if float(color_key).is_integer() else color_key}°" if ori < 180 else None
        ax_ori.scatter(sub["Time_sec"], sub["Orientation"], s=18, color=ori_color_map.get(color_key, "gray"), alpha=0.7, edgecolors="none", label=label)

    handles, labels = ax_ori.get_legend_handles_labels()
    seen = set()
    uniq_handles = []
    uniq_labels = []
    for handle, label in zip(handles, labels):
        if label and label not in seen:
            seen.add(label)
            uniq_handles.append(handle)
            uniq_labels.append(label)
    ax_ori.legend(uniq_handles, uniq_labels, bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True, fontsize=10, title="Orientation")
    yticks = sorted(ori_df["Orientation"].dropna().unique())
    ax_ori.set_yticks(yticks)
    ax_ori.set_yticklabels([f"{int(y) if float(y).is_integer() else y}°" for y in yticks])
    format_axis(ax_ori, title="Grating Orientations Over Time", ylabel="Orientation (degrees)")

    for oddball_type in oddball_order:
        sub = oddball_df[oddball_df["Oddball_Type"] == oddball_type]
        ax_odd.scatter(sub["Time_sec"], [oddball_type] * len(sub), s=42, color=oddball_colors[oddball_type], alpha=0.85, edgecolors="none", label=oddball_type)

    ax_odd.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True, fontsize=10, title="Oddball type")
    format_axis(ax_odd, title="Oddball Events Distribution", xlabel="Time (seconds)", ylabel="Oddball Type")

    xmin = df["Time_sec"].min()
    xmax = block_df["end_sec"].max()
    for ax in [ax_block, ax_ori, ax_odd]:
        ax.set_xlim(xmin, xmax)
        ax.grid(False)

    fig.subplots_adjust(right=0.82)
    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_dir / "task_structure_overview.pdf", dpi=300, bbox_inches="tight")
    plt.show()


# Session Setup


In [ ]:
SESSION_HINT = "MC19"
SEARCH_ROOTS = [Path.home() / "MC19_Processed_New"]
SESSION_INDEX = -1
MANUAL_SESSION_DIR = None
MANUAL_STIM_CSV = None
MANUAL_VOLTAGE_CSV = None

session_info = discover_session_files(
    session_hint=SESSION_HINT,
    search_roots=SEARCH_ROOTS,
    manual_session_dir=MANUAL_SESSION_DIR,
    manual_stim_csv=MANUAL_STIM_CSV,
    manual_voltage_csv=MANUAL_VOLTAGE_CSV,
    session_index=SESSION_INDEX,
)

numeric_cols = [
    "Block_Number", "Block_Duration_Minutes", "Trial_Number", "Sequence_Number", "Trial_In_Sequence",
    "Contrast", "Delay", "DiameterX", "DiameterY", "Duration", "Orientation",
    "Spatial_Frequency", "Temporal_Frequency", "X", "Y",
]

stim_table = pd.read_csv(session_info["stim_csv"])
stim_table = ensure_numeric(stim_table, numeric_cols)
stim_table = add_time_columns(stim_table)
block_df = build_block_summary(stim_table)

figure_dir = session_info["session_dir"] / "Figures"
figure_dir.mkdir(exist_ok=True, parents=True)

print(f"Allen task reference: {ALLEN_DOC_URL}")
print(f"Session directory: {session_info['session_dir']}")
print(f"ops path: {session_info['ops_path']}")
print(f"Stimulus CSV: {session_info['stim_csv']}")
print(f"Voltage CSV: {session_info['voltage_csv']}")
print(f"Session files present: {', '.join(session_info['session_candidate']['present'])}")

display(block_df[["Block_Number", "Block_Label", "Block_Type", "actual_duration_min", "reported_duration_min", "duration_diff_min", "n_trials"]])
plot_task_structure(stim_table, block_df, save_dir=figure_dir)


# DFF Data and FOV


In [ ]:
def create_memmap(data, dtype, mmap_path):
    memmap_arr = np.memmap(mmap_path, dtype=dtype, mode="w+", shape=data.shape)
    memmap_arr[:] = data[...]
    return memmap_arr


def get_memmap_path(session_dir, file_name):
    mm_folder_name, _ = os.path.splitext(file_name)
    mm_path = Path(session_dir) / "memmap" / mm_folder_name
    mm_path.mkdir(parents=True, exist_ok=True)
    file_path = Path(session_dir) / file_name
    return mm_path, file_path


def read_ops(list_session_data_path):
    list_ops = []
    for session_data_path in list_session_data_path:
        session_data_path = Path(session_data_path)
        ops = np.load(resolve_ops_path(session_data_path), allow_pickle=True).item()
        ops["save_path0"] = str(session_data_path)
        list_ops.append(ops)
    return list_ops


def build_masks_from_suite2p(session_dir, ops):
    session_dir = Path(session_dir)
    stat_candidates = [
        session_dir / "suite2p" / "plane0" / "stat.npy",
        session_dir / "qc_results" / "stat.npy",
    ]
    stat_path = next((p for p in stat_candidates if p.exists()), None)
    if stat_path is None:
        raise FileNotFoundError(f"Could not find Suite2p stat.npy for mask reconstruction in {session_dir}")

    stat = np.load(stat_path, allow_pickle=True)
    ly = int(ops.get("Ly", ops["meanImg"].shape[0]))
    lx = int(ops.get("Lx", ops["meanImg"].shape[1]))
    label_img = np.zeros((ly, lx), dtype=np.int32)

    for roi_idx, roi in enumerate(stat, start=1):
        ypix = np.asarray(roi["ypix"], dtype=np.intp)
        xpix = np.asarray(roi["xpix"], dtype=np.intp)
        valid = (ypix >= 0) & (ypix < ly) & (xpix >= 0) & (xpix < lx)
        label_img[ypix[valid], xpix[valid]] = roi_idx

    return label_img


def read_masks(ops):
    session_dir = Path(ops["save_path0"])
    h5_path = session_dir / "masks.h5"

    if h5_path.exists():
        mm_path, file_path = get_memmap_path(session_dir, "masks.h5")
        with h5py.File(file_path, "r") as f:
            labels = create_memmap(f["labels"], "int32", mm_path / "labels.mmap")
            masks = create_memmap(f["masks_func"], "float32", mm_path / "masks_func.mmap")
            mean_func = create_memmap(f["mean_func"], "float32", mm_path / "mean_func.mmap")
            max_func = create_memmap(f["max_func"], "float32", mm_path / "max_func.mmap")
            mean_anat = create_memmap(f["mean_anat"], "float32", mm_path / "mean_anat.mmap") if ops.get("nchannels", 1) == 2 and "mean_anat" in f else None
        return labels, masks, mean_func, max_func, mean_anat

    npy_path = session_dir / "masks.npy"
    qc_path = session_dir / "qc_results" / "masks.npy"
    file_path = npy_path if npy_path.exists() else qc_path
    if file_path.exists():
        mm_path, _ = get_memmap_path(session_dir, file_path.name)
        labels_np = np.load(file_path, allow_pickle=False)
        labels = create_memmap(labels_np.astype(np.int32, copy=False), "int32", mm_path / "labels.mmap")
        masks = create_memmap(labels_np.astype(np.float32, copy=False), "float32", mm_path / "masks_func.mmap")
    else:
        mm_path, _ = get_memmap_path(session_dir, "suite2p_stat.npy")
        labels_np = build_masks_from_suite2p(session_dir, ops)
        labels = create_memmap(labels_np.astype(np.int32, copy=False), "int32", mm_path / "labels.mmap")
        masks = create_memmap(labels_np.astype(np.float32, copy=False), "float32", mm_path / "masks_func.mmap")

    mean_func = create_memmap(np.asarray(ops["meanImg"], dtype=np.float32), "float32", mm_path / "mean_func.mmap")
    max_proj = ops.get("max_proj", ops.get("max_proj_chan2", ops["meanImg"]))
    max_func = create_memmap(np.asarray(max_proj, dtype=np.float32), "float32", mm_path / "max_func.mmap")
    mean_anat = create_memmap(np.asarray(ops["meanImg_chan2"], dtype=np.float32), "float32", mm_path / "mean_anat.mmap") if ops.get("nchannels", 1) == 2 and "meanImg_chan2" in ops else None
    return labels, masks, mean_func, max_func, mean_anat


In [ ]:
list_session_data_path = [session_info["session_dir"]]
list_ops = read_ops(list_session_data_path)
labels, masks, mean_func, max_func, mean_anat = read_masks(list_ops[0])


In [ ]:
def visualize_calcium_signal(image, reference_image=None, method="percentile", channel="red", **kwargs):
    img = np.asarray(image, dtype=float)
    if method == "percentile":
        p_low = kwargs.get("percentile_low", 1)
        p_high = kwargs.get("percentile_high", 99)
        ref = np.asarray(reference_image if reference_image is not None else img, dtype=float)
        low = np.percentile(ref, p_low)
        high = np.percentile(ref, p_high)
        img = np.clip((img - low) / (high - low + 1e-9), 0, 1)

    rgb_img = np.zeros((*img.shape, 3), dtype=np.float32)
    if channel == "red":
        rgb_img[..., 0] = img
    elif channel == "green":
        rgb_img[..., 1] = img
    return rgb_img


def _get_category_mask(masks, labels, cate):
    masks = np.asarray(masks)
    labels = np.asarray(labels)

    if masks.ndim == 3:
        masks = masks[..., 0]
    masks = np.rint(masks).astype(np.int32)

    if labels.shape == masks.shape:
        labels_2d = np.rint(labels).astype(np.int32)
        if np.array_equal(labels_2d, masks):
            return masks > 0 if cate == 0 else np.zeros_like(masks, dtype=bool)
        return labels_2d == cate

    labels = labels.squeeze()
    roi_ids = np.unique(masks)
    roi_ids = roi_ids[roi_ids != 0]

    if labels.ndim != 1:
        raise ValueError(f"Unexpected labels shape: {labels.shape}")

    if len(labels) == len(roi_ids):
        selected_roi_ids = roi_ids[labels == cate]
    elif len(labels) > masks.max():
        selected_roi_ids = [roi_id for roi_id in roi_ids if labels[int(roi_id)] == cate]
    else:
        raise ValueError("1D labels length must match n_rois or max_roi_id + 1.")

    return np.isin(masks, selected_roi_ids)


def thicken_boundary(mask, iterations=2, mode="outer"):
    boundary = find_boundaries(mask, mode=mode)
    return binary_dilation(boundary, iterations=iterations)


def create_superimposed_mask_images(mean_func, max_func, masks, labels, mean_anat=None, boundary_iterations=2):
    mean_func = np.asarray(mean_func)
    max_func = np.asarray(max_func)
    masks = np.asarray(masks)

    if masks.ndim == 3:
        masks = masks[..., 0]
    masks = np.rint(masks).astype(np.int32)

    mean_fun_channel = visualize_calcium_signal(mean_func, method="percentile", channel="green")
    max_fun_channel = visualize_calcium_signal(max_func, method="percentile", channel="green")

    inhibitory_mask = _get_category_mask(masks, labels, 1)
    excitatory_mask = _get_category_mask(masks, labels, -1)
    unsure_mask = _get_category_mask(masks, labels, 0)

    boundary_image = np.zeros((*masks.shape, 3), dtype=np.float32)
    boundary_image[thicken_boundary(inhibitory_mask, iterations=boundary_iterations)] = [1, 0, 1]
    boundary_image[thicken_boundary(excitatory_mask, iterations=boundary_iterations + 1, mode="thick")] = [0, 0.35, 1]
    boundary_image[thicken_boundary(unsure_mask, iterations=boundary_iterations)] = [1, 1, 1]

    superimpose_mask_func = np.clip(np.asarray(mean_fun_channel, dtype=np.float32) + boundary_image, 0, 1)

    if mean_anat is not None:
        mean_anat_channel = visualize_calcium_signal(mean_anat, method="percentile", channel="red")
        superimpose_mask_anat = np.clip(np.asarray(mean_anat_channel, dtype=np.float32) + boundary_image, 0, 1)
    else:
        mean_anat_channel = None
        superimpose_mask_anat = None

    return mean_fun_channel, max_fun_channel, mean_anat_channel, superimpose_mask_func, superimpose_mask_anat


def plot_fov_summary(
    mean_fun_channel,
    max_fun_channel,
    masks,
    superimpose_mask_func,
    mean_anat_channel,
    superimpose_mask_anat,
    save_path=None,
):
    fig = plt.figure(figsize=(24, 12))
    gs = gridspec.GridSpec(2, 4, figure=fig, width_ratios=[1, 1, 1, 1.2], hspace=0.15, wspace=0.1)

    axes = [
        fig.add_subplot(gs[0, 0]),
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[0, 2]),
        fig.add_subplot(gs[0, 3]),
        fig.add_subplot(gs[1, 0]),
        fig.add_subplot(gs[1, 3]),
    ]
    for ax in axes:
        ax.axis("off")
        ax.grid(False)

    axes[0].imshow(mean_fun_channel)
    axes[0].set_title("Mean Functional Channel")

    axes[1].imshow(max_fun_channel)
    axes[1].set_title("Max Functional Channel")

    axes[2].imshow(np.asarray(masks).squeeze(), cmap="gray")
    axes[2].set_title("Masks")

    axes[3].imshow(superimpose_mask_func)
    axes[3].set_title("Mask Overlay on Functional Mean")

    if mean_anat_channel is not None:
        axes[4].imshow(mean_anat_channel)
        axes[4].set_title("Mean Anatomical Channel")

    if superimpose_mask_anat is not None:
        axes[5].imshow(superimpose_mask_anat)
        axes[5].set_title("Mask Overlay on Anatomical Mean")

    legend_handles = [
        Patch(facecolor="#ff00ff", edgecolor="none", label="Inhibitory"),
        Patch(facecolor="#0059ff", edgecolor="none", label="Excitatory"),
        Patch(facecolor="#ffffff", edgecolor="black", label="Unsure"),
    ]
    fig.legend(legend_handles, [h.get_label() for h in legend_handles], bbox_to_anchor=(1.01, 0.98), loc="upper left")
    fig.suptitle("FOV Summary", y=0.99, weight="bold")
    fig.subplots_adjust(right=0.86)

    if save_path is not None:
        save_path = Path(save_path)
        save_path.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path / "FOV_summary.pdf", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
(
    mean_fun_channel,
    max_fun_channel,
    mean_anat_channel,
    superimpose_mask_func,
    superimpose_mask_anat,
) = create_superimposed_mask_images(
    mean_func,
    max_func,
    masks,
    labels,
    mean_anat=mean_anat,
    boundary_iterations=3,
)

plot_fov_summary(
    mean_fun_channel,
    max_fun_channel,
    masks,
    superimpose_mask_func,
    mean_anat_channel,
    superimpose_mask_anat,
    save_path=figure_dir,
)


# Block and Sequence Comparisons


In [ ]:
def read_dff(ops):
    session_dir = Path(ops["save_path0"])
    mm_path, file_path = get_memmap_path(session_dir, "dff.h5")
    with h5py.File(file_path, "r") as f:
        return create_memmap(f["dff"], "float32", mm_path / "dff.mmap")


def read_raw_voltages(ops):
    session_dir = Path(ops["save_path0"])
    raw_h5 = session_dir / "raw_voltages.h5"
    if not raw_h5.exists():
        raise FileNotFoundError(
            "This MC19 processed session does not include raw timing data required for trial/sequence alignment. "
            f"Missing file: {raw_h5}"
        )

    mm_path, file_path = get_memmap_path(session_dir, "raw_voltages.h5")
    with h5py.File(file_path, "r") as f:
        vol_time = create_memmap(f["raw"]["vol_time"], "float32", mm_path / "vol_time.mmap")
        vol_start = create_memmap(f["raw"]["vol_start"], "int8", mm_path / "vol_start.mmap")
        vol_stim_vis = create_memmap(f["raw"]["vol_stim_vis"], "int8", mm_path / "vol_stim_vis.mmap")
        vol_hifi = create_memmap(f["raw"]["vol_hifi"], "int8", mm_path / "vol_hifi.mmap")
        vol_img = create_memmap(f["raw"]["vol_img"], "int8", mm_path / "vol_img.mmap")
        vol_stim_aud = create_memmap(f["raw"]["vol_stim_aud"], "float32", mm_path / "vol_stim_aud.mmap")
        vol_2p_stim = create_memmap(f["raw"]["vol_2p_stim"], "int8", mm_path / "vol_2p_stim.mmap")
        vol_flir = create_memmap(f["raw"]["vol_flir"], "int8", mm_path / "vol_flir.mmap")
        vol_pmt = create_memmap(f["raw"]["vol_pmt"], "int8", mm_path / "vol_pmt.mmap")
        vol_led = create_memmap(f["raw"]["vol_led"], "int8", mm_path / "vol_led.mmap")
    return [
        vol_time,
        vol_start,
        vol_stim_vis,
        vol_img,
        vol_hifi,
        vol_stim_aud,
        vol_2p_stim,
        vol_flir,
        vol_pmt,
        vol_led,
    ]


def zscore_dff(dff, axis=-1, eps=1e-8):
    dff = np.asarray(dff, dtype=np.float32)
    mean = np.nanmean(dff, axis=axis, keepdims=True)
    std = np.nanstd(dff, axis=axis, keepdims=True)
    return (dff - mean) / (std + eps)


def detect_rising_edges(signal, time, threshold=None, min_interval_sec=0.01):
    signal = np.asarray(signal)
    time = np.asarray(time)
    if threshold is None:
        threshold = 0.5 * (np.nanmin(signal) + np.nanmax(signal))
    above = signal > threshold
    rising = np.where(np.diff(above.astype(int)) == 1)[0] + 1
    if len(rising) == 0:
        return np.array([]), np.array([])

    dt = np.median(np.diff(time))
    min_interval_samples = max(1, int(min_interval_sec / dt))
    kept = [rising[0]]
    for idx in rising[1:]:
        if idx - kept[-1] >= min_interval_samples:
            kept.append(idx)
    kept = np.array(kept)
    return time[kept], kept


def normalize_voltage_time(vol_time):
    vol_time = np.asarray(vol_time, dtype=float)
    if np.median(np.diff(vol_time[: min(len(vol_time), 10000)])) > 0.01:
        vol_time = vol_time / 1000.0
    return vol_time


def decode_quadrature_fast(enc_a, enc_b, time_sec, threshold_a=None, threshold_b=None):
    enc_a = np.asarray(enc_a)
    enc_b = np.asarray(enc_b)
    time_sec = np.asarray(time_sec)

    if threshold_a is None:
        threshold_a = 0.5 * (np.nanmin(enc_a) + np.nanmax(enc_a))
    if threshold_b is None:
        threshold_b = 0.5 * (np.nanmin(enc_b) + np.nanmax(enc_b))

    A = enc_a > threshold_a
    B = enc_b > threshold_b
    state = (A.astype(np.uint8) << 1) | B.astype(np.uint8)

    lut = np.zeros(16, dtype=np.float32)
    lut[(0 << 2) | 1] = +1
    lut[(1 << 2) | 3] = +1
    lut[(3 << 2) | 2] = +1
    lut[(2 << 2) | 0] = +1
    lut[(0 << 2) | 2] = -1
    lut[(2 << 2) | 3] = -1
    lut[(3 << 2) | 1] = -1
    lut[(1 << 2) | 0] = -1

    transition_code = (state[:-1] << 2) | state[1:]
    steps = np.zeros(len(state), dtype=np.float32)
    steps[1:] = lut[transition_code]
    position_ticks = np.cumsum(steps)
    dt = np.gradient(time_sec)
    speed_ticks_s = np.gradient(position_ticks) / dt
    return position_ticks, speed_ticks_s


def load_running_speed(voltage_csv, frame_times, n_frames):
    if voltage_csv is None or not Path(voltage_csv).exists():
        warnings.warn("Voltage CSV for wheel decoding was not found; running plots will be skipped.")
        return None

    df_vol = pd.read_csv(voltage_csv, engine="python")
    enc_a = df_vol[" Input 6"].to_numpy()
    enc_b = df_vol[" Input 7"].to_numpy()
    time_sec = df_vol["Time(ms)"].to_numpy(dtype=float) / 1000.0
    _, speed_ticks_s = decode_quadrature_fast(enc_a, enc_b, time_sec)
    speed_ticks_s_smooth = gaussian_filter1d(speed_ticks_s, sigma=20)
    speed_frame = np.interp(frame_times, time_sec, speed_ticks_s_smooth)
    return speed_frame[:n_frames]


def attach_absolute_times(stim_table, frame_times, stim_onset_times, dff):
    if len(stim_onset_times) == 0:
        raise RuntimeError("No visual stimulus onset edge was detected in vol_stim_vis.")
    session_start = stim_onset_times[0]
    stim_table = stim_table.copy()
    stim_table["Abs_Onset_sec"] = session_start + stim_table["Time_sec"].to_numpy(float)
    stim_table["Abs_End_sec"] = stim_table["Abs_Onset_sec"] + stim_table["Duration"].fillna(0).to_numpy(float)
    stim_table["Frame_Index"] = np.searchsorted(frame_times, stim_table["Abs_Onset_sec"], side="left")
    valid = (
        (stim_table["Frame_Index"] >= 0)
        & (stim_table["Frame_Index"] < dff.shape[-1])
        & (stim_table["Abs_Onset_sec"] <= frame_times[-1])
    )
    return stim_table.loc[valid].copy()


def compute_population_alignment(signal, event_frames, win_frames, reducer="mean"):
    signal = np.asarray(signal)
    aligned = np.stack([signal[..., frame + win_frames] for frame in event_frames], axis=0)
    if signal.ndim == 2:
        if reducer == "mean":
            aligned = aligned.mean(axis=1)
        elif reducer == "median":
            aligned = np.median(aligned, axis=1)
    return aligned


def get_window(frame_times, pre_sec, post_sec):
    frame_dt = np.median(np.diff(frame_times))
    fs = 1.0 / frame_dt
    pre_frames = int(round(pre_sec * fs))
    post_frames = int(round(post_sec * fs))
    win_frames = np.arange(-pre_frames, post_frames + 1)
    win_time = win_frames / fs
    return fs, win_frames, win_time, pre_frames, post_frames


def align_trials(signal, table, frame_col, frame_times, pre_sec=1.0, post_sec=1.5, reducer="mean"):
    fs, win_frames, win_time, pre_frames, post_frames = get_window(frame_times, pre_sec, post_sec)
    event_frames = table[frame_col].to_numpy(dtype=int)
    valid = (
        (event_frames - pre_frames >= 0)
        & (event_frames + post_frames < signal.shape[-1])
    )
    valid_table = table.loc[valid].copy()
    valid_frames = event_frames[valid]
    if len(valid_frames) == 0:
        return None
    aligned = compute_population_alignment(signal, valid_frames, win_frames, reducer=reducer)
    return {
        "aligned": aligned,
        "table": valid_table,
        "win_time": win_time,
        "fs": fs,
        "pre_sec": pre_sec,
        "post_sec": post_sec,
    }


def add_stimulus_box(ax, start_sec, duration_sec, color, alpha=0.14, label=None):
    ymin, ymax = ax.get_ylim()
    rect = Rectangle(
        (start_sec, ymin),
        duration_sec,
        ymax - ymin,
        facecolor=color,
        edgecolor="none",
        alpha=alpha,
        zorder=0,
        label=label,
    )
    ax.add_patch(rect)
    ax.set_ylim(ymin, ymax)


def summarize_condition_alignment(result):
    aligned = np.asarray(result["aligned"])
    mean_trace = aligned.mean(axis=0)
    sem_trace = aligned.std(axis=0, ddof=1) / np.sqrt(aligned.shape[0]) if aligned.shape[0] > 1 else np.zeros_like(mean_trace)
    duration_sec = result["table"]["Duration"].median()
    return mean_trace, sem_trace, duration_sec


def plot_two_condition_alignment(results, title, ylabel, save_name, colors=None):
    colors = colors or ["#1f77b4", "#d62728"]
    fig = plt.figure(figsize=(10, 5))
    gs = gridspec.GridSpec(1, 1, figure=fig)
    ax = fig.add_subplot(gs[0])
    legend_handles = []

    for color, (label, result) in zip(colors, results.items()):
        if result is None:
            continue
        mean_trace, sem_trace, duration_sec = summarize_condition_alignment(result)
        ax.plot(result["win_time"], mean_trace, color=color, lw=2.2, label=f"{label} (n={len(result['table'])})")
        ax.fill_between(
            result["win_time"],
            mean_trace - sem_trace,
            mean_trace + sem_trace,
            color=color,
            alpha=0.22,
        )
        legend_handles.append(Patch(facecolor=color, alpha=0.22, edgecolor="none", label=f"{label} mean ± SEM"))
        add_stimulus_box(ax, 0, duration_sec, color=color, alpha=0.10)

    ax.axvline(0, color="black", linestyle="--", lw=1.2)
    format_axis(ax, title=title, xlabel="Time from trial onset (s)", ylabel=ylabel)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True)
    fig.subplots_adjust(right=0.8)
    fig.savefig(figure_dir / save_name, dpi=300, bbox_inches="tight")
    plt.show()


def annotate_sequence_boxes(ax, trial_box_df, oddball_color="#d62728", control_color="#9ecae1"):
    ymin, ymax = ax.get_ylim()
    for _, row in trial_box_df.iterrows():
        color = oddball_color if row["is_oddball"] else control_color
        rect = Rectangle(
            (row["relative_onset_sec"], ymin),
            row["duration_sec"],
            ymax - ymin,
            facecolor=color,
            edgecolor="none",
            alpha=0.10 if row["is_oddball"] else 0.07,
            zorder=0,
        )
        ax.add_patch(rect)
    ax.set_ylim(ymin, ymax)


def build_sequence_tables(stim_table):
    seq_trials = stim_table.dropna(subset=["Sequence_Number"]).copy()
    if "Trial_In_Sequence" not in seq_trials:
        seq_trials["Trial_In_Sequence"] = np.nan
    seq_trials["Trial_In_Sequence"] = seq_trials["Trial_In_Sequence"].fillna(0).astype(int)
    seq_trials["Oddball_Type"] = seq_trials["Trial_Type"].map(oddball_map)

    seq_summary = (
        seq_trials.groupby(["Block_Number", "Sequence_Number"], as_index=False)
        .agg(
            Sequence_Onset_sec=("Abs_Onset_sec", "min"),
            Sequence_End_sec=("Abs_End_sec", "max"),
            Sequence_Start_Frame=("Frame_Index", "min"),
            n_trials=("Trial_Number", "count"),
        )
    )
    seq_summary["Sequence_Duration_sec"] = (
        seq_summary["Sequence_End_sec"] - seq_summary["Sequence_Onset_sec"]
    )
    seq_summary["contains_oddball"] = seq_summary.apply(
        lambda row: seq_trials.loc[
            (seq_trials["Block_Number"] == row["Block_Number"])
            & (seq_trials["Sequence_Number"] == row["Sequence_Number"]),
            "Oddball_Type",
        ].notna().any(),
        axis=1,
    )
    seq_summary["primary_oddball"] = seq_summary.apply(
        lambda row: (
            seq_trials.loc[
                (seq_trials["Block_Number"] == row["Block_Number"])
                & (seq_trials["Sequence_Number"] == row["Sequence_Number"])
                & seq_trials["Oddball_Type"].notna(),
                "Oddball_Type",
            ].iloc[0]
            if seq_trials.loc[
                (seq_trials["Block_Number"] == row["Block_Number"])
                & (seq_trials["Sequence_Number"] == row["Sequence_Number"])
                & seq_trials["Oddball_Type"].notna()
            ].shape[0] > 0
            else "none"
        ),
        axis=1,
    )
    return seq_trials, seq_summary


def sequence_trial_boxes(seq_trials, seq_summary_subset):
    keys = seq_summary_subset[["Block_Number", "Sequence_Number"]].drop_duplicates()
    trial_box_df = seq_trials.merge(keys, on=["Block_Number", "Sequence_Number"], how="inner").copy()
    trial_box_df["relative_onset_sec"] = (
        trial_box_df["Abs_Onset_sec"]
        - trial_box_df.groupby(["Block_Number", "Sequence_Number"])["Abs_Onset_sec"].transform("min")
    )
    trial_box_df["is_oddball"] = trial_box_df["Trial_Type"].isin(oddball_map)
    trial_box_df = (
        trial_box_df.groupby("Trial_In_Sequence", as_index=False)
        .agg(
            relative_onset_sec=("relative_onset_sec", "median"),
            duration_sec=("Duration", "median"),
            is_oddball=("is_oddball", "max"),
        )
        .sort_values("Trial_In_Sequence")
    )
    return trial_box_df


dff = zscore_dff(read_dff(list_ops[0]))

try:
    [
        vol_time,
        vol_start,
        vol_stim_vis,
        vol_img,
        vol_hifi,
        vol_stim_aud,
        vol_2p_stim,
        vol_flir,
        vol_pmt,
        vol_led,
    ] = read_raw_voltages(list_ops[0])
except FileNotFoundError as exc:
    raise RuntimeError(
        "The MC19 processed folder contains stimulus and dF/F data, but not the raw timing channels needed for the "
        "requested alignment figures and Block 1 vs Block 2 event-level comparisons. "
        "You can still use the task-structure and FOV cells, but the alignment section needs raw_voltages.h5 or an "
        "equivalent timing export.\n"
        f"Session: {session_info['session_dir']}\n{exc}"
    )

vol_time_sec = normalize_voltage_time(vol_time)
frame_times, frame_indices = detect_rising_edges(vol_img, vol_time_sec, min_interval_sec=0.005)
stim_onset_times, stim_onset_indices = detect_rising_edges(vol_stim_vis, vol_time_sec, min_interval_sec=0.05)

stim_table_valid = attach_absolute_times(stim_table, frame_times, stim_onset_times, dff)
speed_frame = load_running_speed(session_info["voltage_csv"], frame_times, dff.shape[-1])

print("dff shape:", dff.shape)
print("Detected imaging frames:", len(frame_times))
print("Detected stimulus onset edges:", len(stim_onset_times))
print("Trials inside imaging data:", len(stim_table_valid))


In [ ]:
block1_ori0 = stim_table_valid[
    (stim_table_valid["Block_Number"] == 1)
    & (stim_table_valid["Contrast"] > 0)
    & (stim_table_valid["Orientation_mod180"] == 0)
].copy()

block2_ori0 = stim_table_valid[
    (stim_table_valid["Block_Number"] == 2)
    & (stim_table_valid["Contrast"] > 0)
    & (stim_table_valid["Orientation_mod180"] == 0)
].copy()

block1_oddballs = stim_table_valid[
    (stim_table_valid["Block_Number"] == 1)
    & (stim_table_valid["Trial_Type"].isin(oddball_map))
].copy()

block2_oddballs = stim_table_valid[
    (stim_table_valid["Block_Number"] == 2)
    & (stim_table_valid["Trial_Type"].isin(oddball_map))
].copy()

ori0_dff_results = {
    "Block 1 orientation 0°": align_trials(dff, block1_ori0, "Frame_Index", frame_times, pre_sec=1.0, post_sec=1.5),
    "Block 2 orientation 0°": align_trials(dff, block2_ori0, "Frame_Index", frame_times, pre_sec=1.0, post_sec=1.5),
}
plot_two_condition_alignment(
    ori0_dff_results,
    title="Population 2P Response: Orientation 0° in Block 1 vs Block 2",
    ylabel="z-scored dF/F",
    save_name="block1_vs_block2_orientation0_2p.pdf",
)

if speed_frame is not None:
    ori0_speed_results = {
        "Block 1 orientation 0°": align_trials(speed_frame, block1_ori0, "Frame_Index", frame_times, pre_sec=1.0, post_sec=1.5, reducer="mean"),
        "Block 2 orientation 0°": align_trials(speed_frame, block2_ori0, "Frame_Index", frame_times, pre_sec=1.0, post_sec=1.5, reducer="mean"),
    }
    plot_two_condition_alignment(
        ori0_speed_results,
        title="Running: Orientation 0° in Block 1 vs Block 2",
        ylabel="Wheel speed (ticks/s)",
        save_name="block1_vs_block2_orientation0_running.pdf",
    )

oddball_dff_results = {
    "Block 1 oddballs": align_trials(dff, block1_oddballs, "Frame_Index", frame_times, pre_sec=1.0, post_sec=1.5),
    "Block 2 oddballs": align_trials(dff, block2_oddballs, "Frame_Index", frame_times, pre_sec=1.0, post_sec=1.5),
}
available_oddball_results = {k: v for k, v in oddball_dff_results.items() if v is not None}
if available_oddball_results:
    plot_two_condition_alignment(
        available_oddball_results,
        title="Population 2P Response: Oddballs in Block 1 vs Block 2",
        ylabel="z-scored dF/F",
        save_name="block1_vs_block2_oddballs_2p.pdf",
        colors=["#9467bd", "#ff7f0e"],
    )

if speed_frame is not None and available_oddball_results:
    oddball_speed_results = {
        "Block 1 oddballs": align_trials(speed_frame, block1_oddballs, "Frame_Index", frame_times, pre_sec=1.0, post_sec=1.5, reducer="mean"),
        "Block 2 oddballs": align_trials(speed_frame, block2_oddballs, "Frame_Index", frame_times, pre_sec=1.0, post_sec=1.5, reducer="mean"),
    }
    oddball_speed_results = {k: v for k, v in oddball_speed_results.items() if v is not None}
    plot_two_condition_alignment(
        oddball_speed_results,
        title="Running: Oddballs in Block 1 vs Block 2",
        ylabel="Wheel speed (ticks/s)",
        save_name="block1_vs_block2_oddballs_running.pdf",
        colors=["#9467bd", "#ff7f0e"],
    )


In [ ]:
seq_trials, seq_summary = build_sequence_tables(stim_table_valid)

block1_sequences = seq_summary[
    (seq_summary["Block_Number"] == 1)
    & (~seq_summary["contains_oddball"])
].copy()

block2_sequences = seq_summary[
    (seq_summary["Block_Number"] == 2)
    & (seq_summary["contains_oddball"])
].copy()

def align_sequences(signal, seq_summary_subset, frame_times, pre_sec=0.5, post_sec=1.0, reducer="mean"):
    if seq_summary_subset.empty:
        return None
    median_duration = seq_summary_subset["Sequence_Duration_sec"].median()
    return align_trials(
        signal,
        seq_summary_subset.rename(columns={"Sequence_Start_Frame": "Frame_Index"}),
        "Frame_Index",
        frame_times,
        pre_sec=pre_sec,
        post_sec=post_sec + median_duration,
        reducer=reducer,
    )

seq_dff_results = {
    "Block 1 control sequences": align_sequences(dff, block1_sequences, frame_times, pre_sec=0.5, post_sec=1.5),
    "Block 2 oddball sequences": align_sequences(dff, block2_sequences, frame_times, pre_sec=0.5, post_sec=1.5),
}

fig = plt.figure(figsize=(10, 5))
gs = gridspec.GridSpec(1, 1, figure=fig)
ax = fig.add_subplot(gs[0])
colors = ["#1f77b4", "#d62728"]
for color, (label, result) in zip(colors, seq_dff_results.items()):
    if result is None:
        continue
    mean_trace, sem_trace, _ = summarize_condition_alignment(result)
    ax.plot(result["win_time"], mean_trace, color=color, lw=2.2, label=f"{label} (n={len(result['table'])})")
    ax.fill_between(result["win_time"], mean_trace - sem_trace, mean_trace + sem_trace, color=color, alpha=0.22)

if not block2_sequences.empty:
    annotate_sequence_boxes(ax, sequence_trial_boxes(seq_trials, block2_sequences))
elif not block1_sequences.empty:
    annotate_sequence_boxes(ax, sequence_trial_boxes(seq_trials, block1_sequences), oddball_color="#9ecae1", control_color="#9ecae1")

ax.axvline(0, color="black", linestyle="--", lw=1.2)
format_axis(
    ax,
    title="Sequence-Aligned Population 2P Response: Block 1 Control vs Block 2 Oddball Sequences",
    xlabel="Time from full sequence onset (s)",
    ylabel="z-scored dF/F",
)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True)
fig.subplots_adjust(right=0.8)
fig.savefig(figure_dir / "sequence_aligned_block1_vs_block2_2p.pdf", dpi=300, bbox_inches="tight")
plt.show()

if speed_frame is not None:
    seq_speed_results = {
        "Block 1 control sequences": align_sequences(speed_frame, block1_sequences, frame_times, pre_sec=0.5, post_sec=1.5, reducer="mean"),
        "Block 2 oddball sequences": align_sequences(speed_frame, block2_sequences, frame_times, pre_sec=0.5, post_sec=1.5, reducer="mean"),
    }
    fig = plt.figure(figsize=(10, 5))
    gs = gridspec.GridSpec(1, 1, figure=fig)
    ax = fig.add_subplot(gs[0])
    for color, (label, result) in zip(colors, seq_speed_results.items()):
        if result is None:
            continue
        mean_trace, sem_trace, _ = summarize_condition_alignment(result)
        ax.plot(result["win_time"], mean_trace, color=color, lw=2.2, label=f"{label} (n={len(result['table'])})")
        ax.fill_between(result["win_time"], mean_trace - sem_trace, mean_trace + sem_trace, color=color, alpha=0.22)

    if not block2_sequences.empty:
        annotate_sequence_boxes(ax, sequence_trial_boxes(seq_trials, block2_sequences))
    elif not block1_sequences.empty:
        annotate_sequence_boxes(ax, sequence_trial_boxes(seq_trials, block1_sequences), oddball_color="#9ecae1", control_color="#9ecae1")

    ax.axvline(0, color="black", linestyle="--", lw=1.2)
    format_axis(
        ax,
        title="Sequence-Aligned Running: Block 1 Control vs Block 2 Oddball Sequences",
        xlabel="Time from full sequence onset (s)",
        ylabel="Wheel speed (ticks/s)",
    )
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True)
    fig.subplots_adjust(right=0.8)
    fig.savefig(figure_dir / "sequence_aligned_block1_vs_block2_running.pdf", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
block2_oddball_types = stim_table_valid[
    (stim_table_valid["Block_Number"] == 2)
    & (stim_table_valid["Trial_Type"].isin(oddball_map))
].copy()
block2_oddball_types["Oddball_Type"] = block2_oddball_types["Trial_Type"].map(oddball_map)

dff_type_results = {}
speed_type_results = {}
for oddball_type in oddball_order:
    sub = block2_oddball_types[block2_oddball_types["Oddball_Type"] == oddball_type].copy()
    result = align_trials(dff, sub, "Frame_Index", frame_times, pre_sec=1.0, post_sec=1.5)
    if result is not None:
        dff_type_results[oddball_type] = result
    if speed_frame is not None:
        speed_result = align_trials(speed_frame, sub, "Frame_Index", frame_times, pre_sec=1.0, post_sec=1.5, reducer="mean")
        if speed_result is not None:
            speed_type_results[oddball_type] = speed_result

def plot_multi_condition(results, title, ylabel, save_name):
    fig = plt.figure(figsize=(11, 5))
    gs = gridspec.GridSpec(1, 1, figure=fig)
    ax = fig.add_subplot(gs[0])
    for oddball_type, result in results.items():
        color = oddball_colors[oddball_type]
        mean_trace, sem_trace, duration_sec = summarize_condition_alignment(result)
        ax.plot(result["win_time"], mean_trace, color=color, lw=2.0, label=f"{oddball_type} (n={len(result['table'])})")
        ax.fill_between(result["win_time"], mean_trace - sem_trace, mean_trace + sem_trace, color=color, alpha=0.18)
        add_stimulus_box(ax, 0, duration_sec, color=color, alpha=0.08)
    ax.axvline(0, color="black", linestyle="--", lw=1.2)
    format_axis(ax, title=title, xlabel="Time from trial onset (s)", ylabel=ylabel)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=True)
    fig.subplots_adjust(right=0.8)
    fig.savefig(figure_dir / save_name, dpi=300, bbox_inches="tight")
    plt.show()

if dff_type_results:
    plot_multi_condition(
        dff_type_results,
        title="Population 2P Response: Oddball Types Within Block 2",
        ylabel="z-scored dF/F",
        save_name="block2_oddball_types_2p.pdf",
    )

if speed_type_results:
    plot_multi_condition(
        speed_type_results,
        title="Running: Oddball Types Within Block 2",
        ylabel="Wheel speed (ticks/s)",
        save_name="block2_oddball_types_running.pdf",
    )
